Import modules we need

In [1]:
from utils import *
from google.transit import gtfs_realtime_pb2
ROOT = Path("../")
ROOT.resolve()

PosixPath('/Users/lukestrange/Code/bus-tracking')

We need to unzip the GTFS timetable file to get all the route info. We'll unzip the zip file and store everything in `extract_dir`.

In [2]:
GTFS_file = ROOT / "18SepGB_GTFS_Timetables_Downloaded/itm_yorkshire_gtfs.zip"
# Specify the path to your zip file
zip_file_path = ROOT / "18SepGB_GTFS_Timetables_Downloaded/itm_yorkshire_gtfs.zip"

# Define the directory where you want to extract files
extract_dir = ROOT / "18SepGB_GTFS_Timetables_Downloaded/yorkshire"

unzip_file(zip_file_path, extract_dir)

Successfully unzipped all files in ../18SepGB_GTFS_Timetables_Downloaded/itm_yorkshire_gtfs.zip to ../18SepGB_GTFS_Timetables_Downloaded/yorkshire.


### Dictionaries
To be able to match the live buses to a trip/route etc, we need a way to lookup a given bus's `trip_id` and find its timetabled information. We'll do this using dictionaries because they have [O(1) lookup time thanks to hashmaps](https://dev.to/ajipelumi/how-dictionary-lookup-operations-are-o1-49pk).

First we load just the parts we need into pandas dataframes, then use the `to_dict()` method to create our dictionaries for various ID->ID combinations.

We can get more detailed information about UK stops from the [ffff](). We need the bearing to help match buses to stops on the correct side of the road for their direction of travel.

In [3]:
stop_names_bearings = pd.read_csv(ROOT / "uk_stops/stops.csv", low_memory=False, usecols=['ATCOCode', 'CommonName', 'Bearing'])
stop_names_bearings.rename(columns={"ATCOCode": "stop_id", "CommonName": "stop_name"}, inplace=True)
stop_names_bearings['Bearing'] = stop_names_bearings['Bearing'].map(pd.Series({"N": 0, "NE": 45, "E": 90, "SE": 135, "S": 180, "SW": 225, "W": 270, "NW": 315}))
# What to do about NA Bearings!?
stop_names_bearings[stop_names_bearings.Bearing.notna()].head()

,stop_id,stop_name,Bearing
0,0100BRP90310,Temple Meads Stn,90.0
1,0100BRP90311,Temple Meads Stn,315.0
2,010000056,Temple Meads Stn,0.0
3,0100BRP90312,Temple Meads Stn,45.0
4,0100BRP90313,Temple Meads Stn,45.0


In [4]:
# Load the ID parts of various files that we extracted from the GTFS timetable.
agencies, routes, stops, stop_times, trips = load_gtfs_ids(extract_dir)

# Make the dataframes to be used to make dicts
agency2route = agencies.merge(routes, on='agency_id').set_index('agency_id')
AgencyNOC2AgencyIDDict = agency2route['agency_noc'].drop_duplicates().to_dict()

## Routes have multiple associated trips. Need to think about this. 
## Not sure I am using this anywhere so maybe just remove
# route2trip = routes.merge(trips, on='route_id').set_index('route_id')
# RouteID2TripIDDict = route2trip['trip_id'].to_dict()

# Each trip_id has multiple associated stops. Aggregate these into a list per trip_id.
trip2stoptimes = trips.merge(stop_times, on='trip_id')[['trip_id', 'stop_id']].groupby('trip_id').agg(list)
Trip2StopIDDict = trip2stoptimes['stop_id'].to_dict()

# There are some stop_id in stops.txt that are not in stop_times.txt. 
# We can't use these. Our merge removes these as we complete an "inner" join.
stoptimes2stops = stop_times.merge(stops, on='stop_id').merge(stop_names_bearings, how='inner', on='stop_id')
stoptimes2stops.drop_duplicates(subset='stop_id', keep='first', inplace=True) # We only need unique stop_ids
stoptimes2stops.set_index('stop_id', inplace=True)
StopID2StopLocDict = stoptimes2stops[['stop_lat', 'stop_lon', 'Bearing']].to_dict(orient='index')
# print(len(stops.stop_id.unique()))

Loaded agency.txt


Loaded stop_times.txt
Loaded trips.txt
Loaded stops.txt
Loaded routes.txt


To check this worked properly, we can see if the length of the dictionary is the same as the number of unique IDs from the original GTFS data.

In [5]:
assert len(agencies.agency_id.unique()) == len(AgencyNOC2AgencyIDDict)
# assert len(routes.route_id.unique()) == len(RouteID2TripIDDict)
assert len(trips.trip_id.unique()) == len(Trip2StopIDDict)
# assert len(stoptimes2stops.stop_id.unique()) == len(StopID2StopLocDict)

### Get the paths of the GTFS-RT files.

For each of the extracted GTFS-RT files we get their full paths and save it in `gtfs_rt_file_paths`. This is to save some compute time downstream.

In [6]:
# Directory of the GTFS-RT file
gtfs_rt_dir = ROOT / 'extracted_19SepGB_BusLocations_GTFSRT' 

# Create an empty list to store file paths
gtfs_rt_file_paths = []

# Walk through the directory
for root, dirs, files in os.walk(gtfs_rt_dir):
    for file in files:
        # Get the full path of the file and append it to the list
        full_path = os.path.abspath(os.path.join(root, file))
        gtfs_rt_file_paths.append(full_path)
        

### Load the feed entities 
We load each feed entity into a list (array in most other languages) to loop through later. We could write this in one big loop, but we've split it up to save compute time and make the code easier to follow/debug.

In [7]:
feed = gtfs_realtime_pb2.FeedMessage()
entities = []
for gtfsrt in gtfs_rt_file_paths:
    # Read the GTFS-RT file
    # Ensure we're reading a binary file and not the readme.md or anything else accidentally.
    if '.bin' in str(gtfsrt):
        with open(gtfsrt, 'rb') as file:
            feed.ParseFromString(file.read())
            for entity in feed.entity:
                entities.append(entity)

In [8]:
# no_trip_locs = []
# no_route_locs = []
# for entity in entities:
#     trip = entity.vehicle.trip
#     pos = entity.vehicle.position
#     if trip.trip_id == '':
#         print(pos)
#         no_trip_locs.append((pos.latitude, pos.longitude))
#     if trip.route_id == '':
#         no_route_locs.append((pos.latitude, pos.longitude))
# no_trip_ids = pd.DataFrame(no_trip_locs, columns=['latitude', 'longitude'])
# no_route_ids = pd.DataFrame(no_route_locs, columns=['latitude', 'longitude'])

### Getting the real-time locations of Buses

Now we want to iterate through each minute of the GTFS-RT feed data.
- For each feed message, we iterate through every entity. 
- A single entity contains information about a single trip (bus). 
- Per entity, we instantiate a `Class` called `BusDetail` and save the current trip info to that class.
- If the trip_id exists, we find all the stops on that route using the trip_id.
- Get the locations of the stops on that route.
- Calculate a bounding box...
- Check the bearing using... 
- Calculate the distance to each stop in the simplified list of stops using the Haversine formula.
- Get the `stop_id` and distance of the nearest stop (smallest distance)
- If the nearest stop exists and is closer than 200m, we add this bus to our bag, `BusDetailsBag` for later.

In [9]:
BusDetailsBag = []
count = 0
t0 = time.time()
# For the moment assuming that `vehicle` is passed. there are other mutually exclusive 
# options: 'trip_update', 'alert', and 'shape'. See the DOCS https://gtfs.org/documentation/realtime/reference/#message-feedentity
for entity in entities:
    BD = BusDetail()
    # These are the two main parts of entity
    vehicle = entity.vehicle
    # # These are sub parts
    trip = vehicle.trip
    pos = vehicle.position
    # These are individual values
    BD.feed_uid = entity.id
    BD.trip_id = trip.trip_id
    BD.route_id = trip.route_id
    BD.lat = round(pos.latitude, 6)
    BD.lon = round(pos.longitude, 6)
    BD.bearing = pos.bearing
    BD.ts = vehicle.timestamp
    BD.v_id = vehicle.vehicle.id
    BD.occupancy_status = vehicle.occupancy_status
    BD.current_stop_sequence = vehicle.current_stop_sequence
    BD.current_status = vehicle.current_status

    if BD.trip_id:
        stops_on_route = Trip2StopIDDict.get(BD.trip_id)
        # If stops isn't none - i.e. if this trip ID is part of the WY timetable
        if stops_on_route:
            # Get the stop_id, lat, and lon for each stop on the route.
            # Scaling the longitudes because they get closer together nearer the poles. Could replace BD.lat with cos(~53) for UK average
            actual_stop_locations_on_route = [
                (stopid, StopID2StopLocDict[stopid]['stop_lat'], 
                 StopID2StopLocDict[stopid]['stop_lon'], 
                 abs(StopID2StopLocDict[stopid]['Bearing'] - BD.bearing), 
                 abs(StopID2StopLocDict[stopid]['stop_lat'] - BD.lat), 
                 abs((StopID2StopLocDict[stopid]['stop_lon'] - BD.lon)/np.cos(BD.lat))
                ) for stopid in stops_on_route
            ]
            
            box_size = 0.003 # 0.01 is ~1.1km 
            # @TODO ADD BUS BEARING HERE. Can filter out all stops that are pointing the "wrong direction". Absolute difference between stop bearing and bus bearing should be < 90 degrees. This gives a semi-circle's worth of error margin.
            
            stops_in_bounds = [item for item in actual_stop_locations_on_route if (item[4] < box_size) and (item[5] < box_size) and (item[3] < 90) and (item[3] != 'nan')]
            
            if len(stops_in_bounds) > 0:
                candidate_stops_and_distances = [item + (haversine(BD.lat, BD.lon, item[1], item[2]),) for item in stops_in_bounds]
    
                n_possible_stops = len(candidate_stops_and_distances)
                if n_possible_stops == 1:
                    # Found the nearest stop already. Get the distance and stop_id.
                    closest_stop_id = candidate_stops_and_distances[0][0]
                    closest_stop_distance = candidate_stops_and_distances[0][6]
                else:
                    # Need to get min distance from n stops.
                    index_of_smallest_distance = min(range(len(candidate_stops_and_distances)), key=lambda i: candidate_stops_and_distances[i][6])
                    closest_stop_id = candidate_stops_and_distances[index_of_smallest_distance][0]
                    closest_stop_distance = candidate_stops_and_distances[index_of_smallest_distance][6]
                
                # Add the details to the current bus.
                BD.NearestStopOnRoute = closest_stop_id
                BD.NearestStopDistance = closest_stop_distance * 1e3 #convert to m

                # Ensure we found a nearest stop and that the bus is "sufficiently" close.
                if BD.NearestStopOnRoute != None and BD.NearestStopDistance < 200:
                    BusDetailsBag.append(BD)

    # Creating some info to see progress
    if count!= 0 and count % 500000 == 0:
        t1 = time.time()
        print('Time elapsed:', round(t1-t0, 3), 's')
        print(f"{count} of {len(entities)} entities parsed.")
    count +=1

Time elapsed: 2.272 s
500000 of 29248187 entities parsed.
Time elapsed: 4.486 s
1000000 of 29248187 entities parsed.
Time elapsed: 6.679 s
1500000 of 29248187 entities parsed.
Time elapsed: 9.571 s
2000000 of 29248187 entities parsed.
Time elapsed: 12.122 s
2500000 of 29248187 entities parsed.
Time elapsed: 14.402 s
3000000 of 29248187 entities parsed.
Time elapsed: 16.636 s
3500000 of 29248187 entities parsed.
Time elapsed: 18.942 s
4000000 of 29248187 entities parsed.
Time elapsed: 21.178 s
4500000 of 29248187 entities parsed.
Time elapsed: 23.587 s
5000000 of 29248187 entities parsed.
Time elapsed: 25.813 s
5500000 of 29248187 entities parsed.
Time elapsed: 28.142 s
6000000 of 29248187 entities parsed.
Time elapsed: 30.4 s
6500000 of 29248187 entities parsed.
Time elapsed: 32.694 s
7000000 of 29248187 entities parsed.
Time elapsed: 34.952 s
7500000 of 29248187 entities parsed.
Time elapsed: 37.179 s
8000000 of 29248187 entities parsed.
Time elapsed: 39.554 s
8500000 of 29248187 enti

Now delete duplicate stops (this also gets rid of long periods where the vehicle is waiting)

Sort the frame by timestamp descending, then keep FIRST of duplicate rows.

In [10]:
data = [{'trip_id': bus.trip_id, "route_id": bus.route_id, 'timestamp': bus.ts, 'nearest_stop_id': bus.NearestStopOnRoute, 'distance': bus.NearestStopDistance} for bus in BusDetailsBag]

# Add the data to a dataframe
df = pd.DataFrame(data)

# Work out the human readable time
df['human_time'] = pd.to_datetime(df['timestamp'], unit='s')

# Timezone is currently BST =  UTC + 1, so need to add 1 hour.
df['uk_bst_time_only'] = (df['human_time'] + pd.Timedelta(hours=1)).dt.strftime('%H:%M:%S')

# Weird rows where they aren't the right date. Binning them
df = df[df.human_time.dt.date == pd.to_datetime('2024-09-19').date()]

# only remove duplictaes that have same stop_id, route_id and nearest_stop_id
df.drop_duplicates(subset=['trip_id', 'route_id', 'nearest_stop_id'], keep='first', inplace=True)

# # For a given trip_id, the arrival_time of (n+1)-th stoptime in sequence must not precede the departure_time of n-th stoptime in sequence in stop_times.txt.
# df.sort_values(by=['timestamp', 'trip_id', 'route_id'], ascending=True, inplace=True)

FilteredOrderedBusLocations = df
FilteredOrderedBusLocations[FilteredOrderedBusLocations.trip_id == 'VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f']

,trip_id,route_id,timestamp,nearest_stop_id,distance,human_time,uk_bst_time_only
63324,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726734990,3290YYA00224,43.01,2024-09-19 08:36:30,09:36:30
66429,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726735173,3290YYA00921,85.87,2024-09-19 08:39:33,09:39:33
74153,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726733795,3290YYA00044,44.96,2024-09-19 08:16:35,09:16:35
177831,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726735233,3290YYA00134,6.95,2024-09-19 08:40:33,09:40:33
363934,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726734632,3290YYA03666,46.67,2024-09-19 08:30:32,09:30:32
367511,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726734576,3290YYA00145,97.07,2024-09-19 08:29:36,09:29:36
393764,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726734318,3290YYA00152,103.84,2024-09-19 08:25:18,09:25:18
522277,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726735892,3290YYA00824,13.01,2024-09-19 08:51:32,09:51:32
537781,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726733553,3290YYA03663,10.44,2024-09-19 08:12:33,09:12:33
549610,VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f,5213,1726734735,3290YYA00099,40.17,2024-09-19 08:32:15,09:32:15


Reading in the original GTFS timetable data. We'll use this to pull all the info about the routes based on the trips we were actually able to match buses for.

In [11]:
# agencies, routes, trips, stops, stop_times, calendar, calendar_dates, feed_info, frequencies, shapes = load_full_gtfs(extract_dir)
for file in os.listdir(extract_dir):
    fp = os.path.join(extract_dir, file)
    print(file)
    data = pd.read_csv(fp, low_memory=False)
    if file == 'agency.txt':
        agencies = data
    if file == 'stops.txt':
        stops = data
    if file == 'stop_times.txt':
        stop_times = data
    if file == 'trips.txt':
        trips = data
    if file == 'calendar.txt':
        calendar = data
    if file == 'calendar_dates.txt':
        calendar_dates = data
    if file == 'routes.txt':
        routes = data
    if file == 'shapes.txt':
        shapes = data
    if file == 'feed_info.txt':
        feed_info = data

agency.txt
calendar_dates.txt
stop_times.txt
frequencies.txt
shapes.txt
trips.txt
feed_info.txt
stops.txt
calendar.txt
routes.txt


Now write this timetable as GTFS. This includes:
- agency.txt
- calendar.txt
- calendar_dates.txt
- routes.txt
- stop_times.txt
- stops.txt 
- trips.txt
- shapes.txt

In [12]:
RealRouteIDs = FilteredOrderedBusLocations['route_id'].astype(int)
RealRoutes = routes[routes['route_id'].isin(RealRouteIDs)]
RealRoutes.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/routes.txt", index=False)
RealRoutes

,route_id,agency_id,route_short_name,route_long_name,route_type
255,101910,OP139,M55,NaN,3
256,6554,OP12112,85,NaN,3
257,12443,OP926,AL4,NaN,3
258,103593,OP4522,C81,NaN,3
259,12614,OP933,X11,NaN,3
...,...,...,...,...,...
1267,94855,OP153,341,NaN,3
1268,94856,OP153,341a,NaN,3
1269,94884,OP153,X3,NaN,3
1270,81788,OP12112,595,NaN,3


In [13]:
RealAgencyIDs = routes['agency_id']
RealAgencies = agencies[agencies['agency_id'].isin(RealAgencyIDs)]
RealAgencies.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/agency.txt", index=False)
RealAgencies

,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone,agency_noc
0,OP564,Megabus,https://www.traveline.info,Europe/London,EN,0141 352 4444,MEGA
1,OP5051,FlixBus,https://www.traveline.info,Europe/London,EN,NaN,FLIX
2,OP1123,South Yorkshire Future Tram,https://www.traveline.info,Europe/London,EN,NaN,SYFT
3,OP666,North Yorkshire County Council,https://www.traveline.info,Europe/London,EN,NaN,NYCC
4,OP245,The Little White Bus,https://www.traveline.info,Europe/London,EN,NaN,UWCO
...,...,...,...,...,...,...,...
77,OP10924,Bee Network,https://www.traveline.info,Europe/London,EN,NaN,BNSM
78,OP676,Coastal and Country Coaches,https://www.traveline.info,Europe/London,EN,NaN,CACC
79,OP1554,Stringers,https://www.traveline.info,Europe/London,EN,NaN,SPMW
80,OP230,Generation Travel,https://www.traveline.info,Europe/London,EN,NaN,GTLT


In [14]:
RealTripIDs = FilteredOrderedBusLocations['trip_id']
RealTrips = trips[trips['trip_id'].isin(RealTripIDs)]
RealTrips.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/trips.txt", index=False)
RealTrips

,route_id,service_id,trip_id,trip_headsign,block_id,shape_id,wheelchair_accessible,vehicle_journey_code
3578,101910,202,VJ67044224ac87f03a284ef7115367c3b463061f36,Plumbley,NaN,RPSP6f43dbe23aa8225f764bb32c83536f04a1568208,0,VJ2
3580,101910,202,VJc2e60b69f72f9a036173811d3379e88ef6b7d117,Plumbley,NaN,RPSP6f43dbe23aa8225f764bb32c83536f04a1568208,0,VJ4
3582,101910,202,VJa7f09df06a99fdf8696722fad467eeb068d8aae7,Plumbley,NaN,RPSP6f43dbe23aa8225f764bb32c83536f04a1568208,0,VJ6
3583,101910,202,VJ513fedf56e8c62054d068b92963da5f341a7c1ed,Plumbley,NaN,RPSP6f43dbe23aa8225f764bb32c83536f04a1568208,0,VJ7
3584,101910,202,VJ87349ef1ee19f7cef5efcebcf237ddc6e17d2135,Plumbley,NaN,RPSP6f43dbe23aa8225f764bb32c83536f04a1568208,0,VJ8
...,...,...,...,...,...,...,...,...
58905,11872,237,VJe4e3cc5f135652e70794daf44385fabb980d033c,Newstead,NaN,NaN,0,vj_1
58906,11872,237,VJd99974661961270ba7c87bd04853e6ea6a09ae7b,Newstead,NaN,NaN,0,vj_5
58907,11872,237,VJb3db91cf5f7fe43fdca670e031912c05c8a45bda,New Crofton,NaN,NaN,0,vj_4
58908,11872,237,VJa64fc62f02937d9698fca2db19436737535bcd5c,Wakefield City Centre,NaN,NaN,0,vj_6


In [15]:
RealShapeIDs = RealTrips['shape_id'].to_list()
RealShapes = shapes[shapes.shape_id.isin(RealShapeIDs)]
RealShapes.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/shapes.txt", index=False)

In [16]:
RealServiceIDs = RealTrips['service_id']
RealCalendar = calendar[calendar['service_id'].isin(RealServiceIDs)]
RealCalendarDates = calendar_dates[calendar_dates['service_id'].isin(RealServiceIDs)]
RealCalendar.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/calendar.txt", index=False)
RealCalendarDates.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/calendar_dates.txt", index=False)

Stop times.txt.

minimally required fields are trip_id, arrival_time and/or departure_time, stop_id, and stop_sequence (must be sequential)

In [17]:
RealTripIDs_list = RealTripIDs.to_list()
FilteredStopTimes = stop_times[stop_times.trip_id.isin(RealTripIDs_list)]
FilteredOrderedBusLocations.rename(columns={'nearest_stop_id': 'stop_id'}, inplace=True)
RealTrips2RealStopsDict = FilteredStopTimes.groupby('trip_id')['stop_id'].agg(list).to_dict()

In [18]:
RealStopTimes = FilteredStopTimes.merge(FilteredOrderedBusLocations, on=['trip_id', 'stop_id'], how='inner')
RealStopTimes['arrival_time'] = RealStopTimes['uk_bst_time_only']
RealStopTimes['departure_time'] = RealStopTimes['uk_bst_time_only']

# A trip must visit more than one stop in stop_times.txt to be usable by passengers for boarding and alighting.
stop_counts = RealStopTimes.groupby('trip_id')['stop_id'].count()
trip_ids_with_one_stop = stop_counts[stop_counts == 1].index.to_list()
RealStopTimes = RealStopTimes[~RealStopTimes.trip_id.isin(trip_ids_with_one_stop)] # Exclude trips with only 1 stop. (The "~" is negation)

# When sorted by stop_times.stop_sequence, two consecutive entries in stop_times.txt 
# should have increasing distance, based on the field shape_dist_traveled. 
# If the values are equal, this is considered as an error.
RealStopTimes.sort_values(by=['trip_id', 'stop_sequence'], ascending=True, inplace=True)

# If pick up type == 1, no pickup is available (Guessing this means you can't get on the bus here?)
RealStopTimes = RealStopTimes[RealStopTimes.pickup_type != 1]

# Remove duplicate rows based on the specified columns but keep the first occurrence
RealStopTimes = RealStopTimes.drop_duplicates(subset=['trip_id', 'stop_id', 'arrival_time', 'departure_time', 'shape_dist_traveled'], keep='first')

# Filtering only columns we need to write stop_times.txt
RealStopTimes = RealStopTimes[['trip_id','arrival_time','departure_time','stop_id','stop_sequence','stop_headsign','pickup_type','drop_off_type','shape_dist_traveled','timepoint']]
# RealStopTimes[RealStopTimes.trip_id == "VJ79ee6abdf8b1015398ec41a7f77b76c7f6a0b10f"]

RealStopTimes.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/stop_times.txt",index=False)

Stops.txt

In [19]:
RealStopIDs = FilteredOrderedBusLocations['stop_id'].to_list()
RealStops = stops[stops.stop_id.isin(RealStopIDs)].copy()

# Location type must be able to be parsed as an integer.
RealStops['location_type'] = RealStops['location_type'].astype('Int64')
RealStops.drop(columns='parent_station', inplace=True)
RealStops.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/stops.txt", index=False)

### Creating and writing `feed_info.txt`

In [20]:
feed_info.to_csv(ROOT / "real-19SepGB_GTFS_Timetables/feed_info.txt", index=False)

In [21]:
RealStopTimes[RealStopTimes.trip_id == 'VJd7d4468657e5d515fd5ff0fe16914a9e5001e074']

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
455323,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,15:55:08,15:55:08,450029340,2,NaN,0,0,NaN,0
455324,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,15:56:03,15:56:03,450019268,3,NaN,0,0,NaN,1
455325,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,15:57:15,15:57:15,450050789,5,NaN,0,0,NaN,0
455326,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,15:58:57,15:58:57,450028418,6,NaN,0,0,NaN,1
455327,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,16:01:44,16:01:44,450019237,9,NaN,0,0,NaN,1
455328,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,16:03:00,16:03:00,450019234,11,NaN,0,0,NaN,0
455329,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,16:04:41,16:04:41,450019232,13,NaN,0,0,NaN,0
455330,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,16:07:22,16:07:22,450019192,16,NaN,0,0,NaN,0
455331,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,16:08:15,16:08:15,450019190,17,NaN,0,0,NaN,1
455332,VJd7d4468657e5d515fd5ff0fe16914a9e5001e074,16:09:20,16:09:20,450019188,19,NaN,0,0,NaN,0


Need to write some code to automaticall zip these files.